# Model check: same excitation, simulator vs models

Drives the MuJoCo simulator (`make_calibrated_sim`, treated as the real robot) and five models
with **the same commands**, then compares the tip per axis:

| Model | Built by | Inputs it takes | Outputs |
|---|---|---|---|
| One-mode model | Nikoletta (latest notebook, Smith-2 §7): ÿ + 9.231 ẏ + 10.60 y = b Δp | plane pressure differences: x ← −u₃, y ← u₂ − u₄ (no actuator dynamics) | x, y |
| One-mode + gravity | Nikoletta's derivation with the chain's gravity stiffness added (our review, not in her notebooks): k = 51.45 | same | x, y |
| M2 | Ibraheem, `task_space_model.py`: first-order mechanics | the three segment commands, through dead time 0.5 s + lag 0.6 s | x, y, z |
| M3 | Ibraheem, `task_space_model.py`: M2's actuator + second-order mechanics (inertia) | same as M2 | x, y, z |
| F | Ibraheem, `sysid_frf.py` | channels ν_x = −u₃, ν_y = u₂ − u₄, with the identified delay | x, y |

u = command − 4.5 psi. Every model runs **open loop from rest** (free run): it never sees the
simulator's tip, so errors accumulate exactly as they would in a controller's internal model.

**Reading the plots without colour.** Each model has its own line style *and* marker
(table below); the simulator is always the thick black dashed line. Plots are split per axis into
two panels, "derived from the simulator's internals" (one-mode models) and "identified from data"
(M2, M3, F), so no panel has more than three models.

| Model | Line | Marker |
|---|---|---|
| Simulator (truth) | thick black, long dashes | none |
| One-mode (Nikoletta) | dashed | square ■ |
| One-mode + gravity | dotted | diamond ◆ |
| M2 | solid | circle ● |
| M3 | dashed | triangle down ▼ |
| F | dash-dot | triangle up ▲ |

All code is in `scripts/compare_models_excitation.py`; this notebook only calls it and plots.
Runs offline in the project environment (`uv run jupyter lab`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import compare_models_excitation as CM
import plot_style as P

P.apply()
%matplotlib inline

## 1 — Choose an excitation

- `"random_steps"`: independent random levels per segment, 1.5–7.5 psi, held 0.5–3 s.
- `"chirp"`: phased sines (0°, 120°, 240°) on S2–S4, 4.5 ± 2.5 psi, frequency swept 0.02 → 1 Hz.
- Or set `CMDS` yourself: any `(n, 3)` array of absolute psi for S2, S3, S4 at 100 Hz, within 0–9.

Seeds used for identification and validation elsewhere: 1–4, 11–14, 21–22, 101. Pick another.

In [ ]:
KIND = "random_steps"     # or "chirp"
SECONDS = 60.0
SEED = 31                 # random_steps only

CMDS = CM.excitation(KIND, SECONDS, seed=SEED)
# CMDS = np.column_stack([...])   # custom excitation: (n, 3) psi, 100 Hz

t = np.arange(len(CMDS)) * CM.DT
fig, ax = plt.subplots(figsize=(P.WIDTH, 3.5))
for j, seg in enumerate(("S2", "S3", "S4")):
    ax.plot(t, CMDS[:, j], **P.line(seg, len(t)))
ax.set(xlabel="time (s)", ylabel="command (psi)", title=f"Excitation: {KIND}")
ax.grid(alpha=0.3); P.legend_outside(ax)
plt.show()

## 2 — Run the simulator and the five models

In [ ]:
u = CMDS - CM.BASE
truth = CM.simulate_truth(CMDS)                  # MuJoCo, seed 0, 5 s settle at 4.5 psi first
sims = {m: CM.SIMULATE[m](u) for m in CM.MODELS}
results = {m: CM.metrics(truth, sims[m]) for m in CM.MODELS}

print(f"{'model':<30}" + "".join(f"{a + ' RMSE (fit)':>22}" for a in "xyz"))
for m in CM.MODELS:
    cells = [f"{v['rmse_mm']:.3f} mm ({v['fit_pct']:5.1f} %)" if v else "no z output" for v in results[m].values()]
    print(f"{CM.LABELS[m]:<30}" + "".join(f"{c:>22}" for c in cells))

**Reading the numbers.** RMSE is in mm. Fit = 100 (1 − ‖model − truth‖ / ‖truth − mean‖): 100 %
is perfect, 0 % is as good as predicting a constant, negative is worse than a constant.

## 3 — Tip per axis

In [ ]:
GROUPS = ((CM.DERIVED, "derived from the simulator's internals"), (CM.IDENTIFIED, "identified from data"))
fig, axs = P.figure(3, 2, sharex=True, layout="constrained")
fig.set_size_inches(P.WIDTH * 1.5, P.PANEL_H * 3)
for i, name in enumerate("xyz"):
    for j, (group, title) in enumerate(GROUPS):
        ax = axs[i, j]
        ax.plot(t, truth[:, i], **{**P.REFERENCE, "label": CM.LABELS["truth"]})
        for m in group:
            if not np.isnan(sims[m][:, i]).all():
                ax.plot(t, sims[m][:, i], **P.line(CM.LABELS[m], len(t)))
        ax.set(ylabel=f"tip {name} − settled (mm)", title=f"{name} axis: {title}")
        ax.grid(alpha=0.3); ax.legend(loc="upper right", fontsize=9)
for ax in axs[-1]:
    ax.set_xlabel("time (s)")
plt.show()

## 4 — Error per axis (model − simulator)

In [ ]:
fig, axs = P.figure(3, 2, sharex=True, layout="constrained")
fig.set_size_inches(P.WIDTH * 1.5, P.PANEL_H * 3)
for i, name in enumerate("xyz"):
    for j, (group, title) in enumerate(GROUPS):
        ax = axs[i, j]
        for m in group:
            if not np.isnan(sims[m][:, i]).all():
                ax.plot(t, sims[m][:, i] - truth[:, i], **P.line(CM.LABELS[m], len(t)))
        ax.axhline(0, color="black", lw=1)
        ax.set(ylabel=f"{name} error (mm)", title=f"{name} axis, model − simulator: {title}")
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc="upper right", fontsize=9)
for ax in axs[-1]:
    ax.set_xlabel("time (s)")
plt.show()

## 5 — Save the data

Writes the commands, the simulator's tip and every model's tip (mm, 100 Hz) to an `.npz` file,
and the metrics to a `.json` file next to it. Load later with `np.load(path)`.

In [ ]:
import json
out = ROOT / "docs" / "figures" / f"model_check_{KIND}_seed{SEED}"
np.savez_compressed(out.with_suffix(".npz"), t=t, cmds=CMDS, truth=truth, **{m: s for m, s in sims.items()})
out.with_suffix(".json").write_text(json.dumps(dict(kind=KIND, seconds=SECONDS, seed=SEED, metrics=results), indent=1))
print("saved", out.with_suffix(".npz"), "and", out.with_suffix(".json"))

## Why the one-mode models are off

Nikoletta's one-mode model moves in the right direction at the right moments but about 4× too far
and with no delay: its static gain is b/k = 7.7 mm/psi (no gravity stiffness in its derivation),
against 2.0 mm/psi in the simulator, and it has neither the 0.5 s dead time nor the 0.6 s pressure
lag. Adding the gravity stiffness brings the gain to 1.5 mm/psi (now 25 % too small) and cuts the
error by about 3×, but it still leads the simulator by about 1.1 s because it has no delay or lag.
M2, M3 and F were identified from the simulator's data with those dynamics in, so they follow it
to about 0.1 mm; M3's inertia term halves M2's error on y.

## Error table: every model, both excitations, every axis

Runs the simulator and all models on both standard excitations (random steps seed 31, chirp;
60 s each) and tabulates, per axis: free-run RMSE, largest |error|, bias = mean error (all mm),
and fit %. Then the static gains (+1 psi step held 10 s from 4.5 psi). **Bold** = best in its
column (smallest RMSE / |max| / |bias|, largest fit, gain closest to the simulator); "–" = the
model has no output on that axis. Takes about a minute.

In [ ]:
from IPython.display import HTML, display

STATS = (("rmse_mm", "RMSE (mm)", lambda v: v, min), ("max_abs_mm", "max |e| (mm)", lambda v: v, min),
         ("bias_mm", "bias (mm)", abs, min), ("fit_pct", "fit (%)", lambda v: v, max))
FMT = {"rmse_mm": "{:.3f}", "max_abs_mm": "{:.2f}", "bias_mm": "{:+.3f}", "fit_pct": "{:.1f}"}
CSS = "<style>.mc td,.mc th{padding:3px 8px;border:1px solid #bbb;text-align:center}.mc td:first-child{text-align:left}</style>"

all_metrics = {}
for kind in ("random_steps", "chirp"):
    cmds = CM.excitation(kind, 60.0, seed=31)
    tr = CM.simulate_truth(cmds)
    all_metrics[kind] = {m: CM.metrics(tr, CM.SIMULATE[m](cmds - CM.BASE)) for m in CM.MODELS}

def shown(key, v):
    """The value as printed, so models that tie on screen are bolded together."""
    return float(FMT[key].format(v))

def best(kind, ax, key, score, pick):
    vals = [score(shown(key, all_metrics[kind][m][ax][key])) for m in CM.MODELS if all_metrics[kind][m][ax]]
    return pick(vals)

html = CSS
for kind in all_metrics:
    head = "<tr><th rowspan=2>model</th>" + "".join(f"<th colspan=4>{a}</th>" for a in "xyz") + "</tr><tr>"
    head += "".join(f"<th>{lab}</th>" for _ in "xyz" for _, lab, _, _ in STATS) + "</tr>"
    rows = ""
    for m in CM.MODELS:
        rows += f"<tr><td>{CM.LABELS[m]}</td>"
        for ax in "xyz":
            v = all_metrics[kind][m][ax]
            for key, _, score, pick in STATS:
                if v is None:
                    rows += "<td>–</td>"
                    continue
                txt = FMT[key].format(v[key])
                rows += f"<td><b>{txt}</b></td>" if score(shown(key, v[key])) == best(kind, ax, key, score, pick) else f"<td>{txt}</td>"
        rows += "</tr>"
    html += f"<h4>Excitation: {kind}</h4><table class=mc>{head}{rows}</table>"

gains = CM.static_gains()
ghead = "<tr><th>static gain (mm/psi)</th><th>simulator</th>" + "".join(f"<th>{CM.LABELS[m]}</th>" for m in CM.MODELS) + "</tr>"
grows = ""
for name, g in gains.items():
    closest = min(CM.MODELS, key=lambda m: abs(g[m] - g["truth"]))
    grows += f"<tr><td>{name}</td><td>{g['truth']:+.2f}</td>" + "".join(
        (f"<td><b>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</b></td>" if m == closest
         else f"<td>{g[m]:+.2f} ({100 * g[m] / g['truth']:.0f} %)</td>")
        for m in CM.MODELS) + "</tr>"
html += f"<h4>Static gain: tip change 10 s after +1 psi (% of the simulator)</h4><table class=mc>{ghead}{grows}</table>"
display(HTML(html))